# THL Data Ingestion Test (Daily)

This notebook uses daily THL source paths and demonstrates:
- fetch -> raw snapshot save
- processed local file save
- reload to ObservedDataset / TimeSeries
- combined plot: cases + hospital load + deaths


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "extensions" / "scenario_api").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate extensions/scenario_api from current working directory")

extensions_dir = repo_root / "extensions"
if str(extensions_dir) not in sys.path:
    sys.path.insert(0, str(extensions_dir))

from scenario_api import (
    download_and_save_thl_cases_snapshot,
    load_processed_table,
    load_thl_cases_observed_dataset,
    thl_dataset_to_timeseries,
    fetch_thl_deaths_raw,
)


In [ ]:
raw_path = repo_root / "data" / "raw" / "thl_cases_2020_2022_raw_daily.csv"
processed_path = repo_root / "data" / "processed" / "thl_cases_2020_2022_processed_daily.csv"
raw_path.parent.mkdir(parents=True, exist_ok=True)
processed_path.parent.mkdir(parents=True, exist_ok=True)
raw_path, processed_path


In [ ]:
saved_raw, saved_processed = download_and_save_thl_cases_snapshot(
    raw_output_path=raw_path,
    processed_output_path=processed_path,
    start_date="2020-01-01",
    end_date="2022-12-31",
    region_level="hospital_district",
)
print("Saved raw snapshot:", saved_raw)
print("Saved processed table:", saved_processed)


In [ ]:
processed = load_processed_table(processed_path)
print("Rows:", len(processed))
print("Date range:", processed["date"].min(), "->", processed["date"].max())
print("Region levels:", sorted(processed["region_level"].unique()))
processed.head()


In [ ]:
dataset = load_thl_cases_observed_dataset(processed_path)
hd_series = thl_dataset_to_timeseries(dataset, variable="cases", region_level="hospital_district")
print("Hospital district series:", len(hd_series))
print("First series:", hd_series[0])
print("First head:", list(zip(hd_series[0].times[:5], hd_series[0].values[:5])))


In [ ]:
from io import StringIO
from urllib.request import urlopen
import pandas as pd

# Cases by hospital district (daily) from processed ingestion
cases_by_area = processed.copy()
cases_by_area["date"] = pd.to_datetime(cases_by_area["date"])
cases_by_area = cases_by_area.sort_values(["region", "date"])

# Deaths by hospital district (daily) using same THL cube measure
deaths_raw = fetch_thl_deaths_raw()
deaths = pd.read_csv(StringIO(deaths_raw), sep=";")
deaths = deaths.rename(columns={"Time": "date", "Area": "region", "val": "value"})
deaths["date"] = pd.to_datetime(deaths["date"], errors="coerce")
deaths["value"] = pd.to_numeric(deaths["value"], errors="coerce").fillna(0.0)
deaths = deaths.dropna(subset=["date", "region"])
deaths = deaths[(deaths["date"] >= "2020-01-01") & (deaths["date"] <= "2022-12-31")].copy()
deaths_by_area = deaths.groupby(["date", "region"], as_index=False)["value"].sum().sort_values(["region", "date"])

# Hospital load by area (daily).
# THL covid19care provides daily area split at ERVA level (5 areas), not hospital district level.
hospital_url = (
    "https://sampo.thl.fi/pivot/prod/en/epirapo/covid19care/"
    "fact_epirapo_covid19care.csv?row=erva-456367&column=dateweek20200101-509038L&"
    "filter=measure-456732&"
)
hospital_raw = urlopen(hospital_url, timeout=60).read().decode("utf-8")
hospital = pd.read_csv(StringIO(hospital_raw), sep=";")
hospital = hospital.rename(columns={"Time": "date", "Area": "region", "val": "value"})
hospital["date"] = pd.to_datetime(hospital["date"], errors="coerce")
hospital["value"] = pd.to_numeric(hospital["value"], errors="coerce").fillna(0.0)
hospital = hospital.dropna(subset=["date", "region"])
hospital = hospital[(hospital["date"] >= "2020-01-01") & (hospital["date"] <= "2022-12-31")].copy()
hospital_by_area = hospital.groupby(["date", "region"], as_index=False)["value"].sum().sort_values(["region", "date"])

print("Cases: daily hospital-district areas =", cases_by_area["region"].nunique())
print("Deaths: daily hospital-district areas =", deaths_by_area["region"].nunique())
print("Hospital load: daily area split =", hospital_by_area["region"].nunique(), "(ERVA level)")


In [ ]:
import io
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

def _plot_by_area(frame, title, ylabel, figsize=(13, 6)):
    fig, ax = plt.subplots(figsize=figsize)
    for area, grp in frame.groupby("region"):
        ax.plot(grp["date"].tolist(), grp["value"].astype(float).tolist(), label=str(area), linewidth=1.0, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.2)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=7, frameon=False)
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig

fig_cases = _plot_by_area(cases_by_area, "Daily cases by hospital district (THL)", "Cases")
fig_deaths = _plot_by_area(deaths_by_area, "Daily deaths by hospital district (THL)", "Deaths")
fig_hospital = _plot_by_area(hospital_by_area, "Daily hospital load by area (THL covid19care, ERVA level)", "Hospital load")

for fig in [fig_cases, fig_deaths, fig_hospital]:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120)
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.read()))
